In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 06:19:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 06:19:38 WARN 

spark session created


In [ ]:
{"id": 1, "price": 100}
{"id": 2, "price": "200"}
{"id": 3, "price": "abc"}
{"id": 4, "price": null}

In [37]:
incsv="s3a://glue-spark-etl-example-source/resolvechoice.csv"
df=spark.read.csv(incsv,inferSchema=True)
df.printSchema()
df.show()
df.filter(col("_c1")=='null').show() # so null is a string not a typically null value here 
print("filter actual NULL values")
df.filter(col("_c1").isNull()).show() # this is how you filter null values



root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)

+---+----+
|_c0| _c1|
+---+----+
|  1| 100|
|  2| 200|
|  3| abc|
|  4|null|
|  5|null|
+---+----+

+---+----+
|_c0| _c1|
+---+----+
|  4|null|
+---+----+

filter actual NULL values
+---+----+
|_c0| _c1|
+---+----+
|  5|null|
+---+----+



In [38]:
df2= df.withColumn("price", col("_c1").cast("int"))
df2.printSchema()
df2.show()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- price: integer (nullable = true)

+---+----+-----+
|_c0| _c1|price|
+---+----+-----+
|  1| 100|  100|
|  2| 200|  200|
|  3| abc| null|
|  4|null| null|
|  5|null| null|
+---+----+-----+



In [39]:
schema = "id int, price int"
df=spark.read.csv(incsv, schema=schema)
df.show()

+---+-----+
| id|price|
+---+-----+
|  1|  100|
|  2|  200|
|  3| null|
|  4| null|
|  5| null|
+---+-----+



In [43]:
input_path="s3a://glue-spark-etl-example-source/resolvechoice_example1.json"
df = spark.read.json(input_path)
df.printSchema()
df.show()

root
 |-- id: long (nullable = true)
 |-- price: string (nullable = true)

+---+--------------+
| id|         price|
+---+--------------+
|  1|           100|
|  2|           200|
|  3|{"amount":300}|
|  4|       ["400"]|
|  5|          null|
|  4|           abc|
+---+--------------+



In [44]:
df1= df.withColumn("price", col("price").cast("int"))
df1.printSchema()
df1.show()

root
 |-- id: long (nullable = true)
 |-- price: integer (nullable = true)

+---+-----+
| id|price|
+---+-----+
|  1|  100|
|  2|  200|
|  3| null|
|  4| null|
|  5| null|
|  4| null|
+---+-----+



In [45]:
schema = "id int, price int"
df=spark.read.json(input_path, schema=schema)
df.show()

+---+-----+
| id|price|
+---+-----+
|  1|  100|
|  2| null|
|  3| null|
|  4| null|
|  5| null|
|  4| null|
+---+-----+



In [61]:
from pyspark.sql.functions import col, when, from_json
from pyspark.sql.types import StructType, StructField, IntegerType

df = spark.read.json(input_path)

df = df.withColumn("price_str", col("price").cast("string"))

# Try parsing struct
schema = StructType([
    StructField("amount", IntegerType(), True)
])

df = df.withColumn("price_struct", from_json(col("price_str"), schema))

df = df.withColumn(
    "price_clean",
    when(col("price_str").cast("int").isNotNull(), col("price_str").cast("int"))
    .when(col("price_struct.amount").isNotNull(), col("price_struct.amount"))
    .when(col("price").getItem(0).cast("int").isNotNull(), col("price")[0].cast("int"))
)

df.show(truncate=False)

AnalysisException: Can't extract value from price#1538: need struct type but got string

## without resolvechoice

In [49]:
# Read JSON from S3
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [input_path]},
    format="json"
)
dyf.show()
dyf.printSchema()

print("===== ORIGINAL DATA =====")
dyf.toDF().show()


{"id": 1, "price": 100}
{"id": 2, "price": "200"}
{"id": 3, "price": {"amount": 300}}
{"id": 4, "price": ["400"]}
{"id": 5}
{"id": 4, "price": "abc"}
root
|-- id: int
|-- price: choice
|    |-- array
|    |    |-- element: string
|    |-- int
|    |-- string
|    |-- struct
|    |    |-- amount: int

===== ORIGINAL DATA =====
+---+--------------------+
| id|               price|
+---+--------------------+
|  1|{null, 100, null,...|
|  2|{null, null, 200,...|
|  3|{null, null, null...|
|  4|{[400], null, nul...|
|  5|                null|
|  4|{null, null, abc,...|
+---+--------------------+



In [50]:
# Apply mapping directly
mapped_dyf = dyf.apply_mapping([
    ("id", "long", "id", "long"),
    ("price", "string", "price", "int")
])

print("===== AFTER APPLYMAPPING ONLY =====")
mapped_dyf.toDF().show()

print("===== NULL COUNT =====")
mapped_dyf.toDF().filter("price IS NULL").show()

===== AFTER APPLYMAPPING ONLY =====
+----+-----+
|  id|price|
+----+-----+
|null| null|
|null|  200|
|null| null|
|null| null|
|null| null|
|null| null|
+----+-----+

===== NULL COUNT =====
+----+-----+
|  id|price|
+----+-----+
|null| null|
|null| null|
|null| null|
|null| null|
|null| null|
+----+-----+



In [53]:
dyf.show()
print("===== AFTER RESOLVE CHOICE =====")
dyf.resolveChoice(specs=[
    ('price', 'cast:int')
]).show()

{"id": 1, "price": 100}
{"id": 2, "price": "200"}
{"id": 3, "price": {"amount": 300}}
{"id": 4, "price": ["400"]}
{"id": 5}
{"id": 4, "price": "abc"}
===== AFTER RESOLVE CHOICE =====
{"id": 1, "price": 100}
{"id": 2, "price": 200}
{"id": 5}
{"id": 4}


In [54]:
dyf.show()
print("===== AFTER RESOLVE CHOICE =====")
dyf.resolveChoice(choice='make_cols').show()

{"id": 1, "price": 100}
{"id": 2, "price": "200"}
{"id": 3, "price": {"amount": 300}}
{"id": 4, "price": ["400"]}
{"id": 5}
{"id": 4, "price": "abc"}
===== AFTER RESOLVE CHOICE =====
{"id": 1, "price_int": 100}
{"id": 2, "price_string": "200"}
{"id": 3, "price_struct": {"amount": 300}}
{"id": 4, "price_array": ["400"]}
{"id": 5}
{"id": 4, "price_string": "abc"}


## with resolvechoice

In [11]:
# Read JSON
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [input_path]},
    format="json"
)

print("===== ORIGINAL DYNAMIC FRAME =====")
dyf.printSchema()

# 🔥 STEP 1: Resolve choice FIRST
resolved_dyf = dyf.resolveChoice(specs=[('price', 'cast:int')])

print("===== AFTER RESOLVECHOICE =====")
resolved_dyf.toDF().show()

# Convert to DataFrame AFTER resolving
df = resolved_dyf.toDF()

# Now filtering works correctly
bad_df = df.filter("price IS NULL")
good_df = df.filter("price IS NOT NULL")

print("===== BAD RECORDS =====")
bad_df.show()

print("===== GOOD RECORDS =====")
good_df.show()

===== ORIGINAL DYNAMIC FRAME =====
root
|-- id: int
|-- price: choice
|    |-- int
|    |-- string

===== AFTER RESOLVECHOICE =====


/home/glue_user/spark/python/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+-----+
| id|price|
+---+-----+
|  1|  100|
|  2|  200|
|  3| null|
|  4| null|
+---+-----+

===== BAD RECORDS =====
+---+-----+
| id|price|
+---+-----+
|  3| null|
|  4| null|
+---+-----+

===== GOOD RECORDS =====
+---+-----+
| id|price|
+---+-----+
|  1|  100|
|  2|  200|
+---+-----+

